# Lab 3 - N-Gram Language Models

Completed task: preprocess the assigned tweets, train a bigram maximum-likelihood model, evaluate it, generate example text, calculate `P(is | pakistan)`, and examine the perplexity of `pakistan`.

Source: [Anonymous - Random Tweets](https://www.kaggle.com/datasets/adizafar/large-random-tweets-from-pakistan), uploaded by adizafar. The corpus includes multiple languages. Unicode letters are retained.

In [1]:
from pathlib import Path
import re
import pandas as pd
import nltk

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "download_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository or one of its lab folders.")
nltk.data.path.insert(0, str(ROOT / ".nltk_data"))

## 1. Load the dataset

The source contains some invalid UTF-8 sequences. Only those sequences are replaced with the Unicode replacement character; the original file is unchanged. The affected tweet count is reported below. Affected tweets are excluded before modeling to avoid learning corrupted word fragments. Valid Unicode text is preserved; decoding the entire file as Latin-1 would corrupt it.

In [2]:
path = ROOT / "Lab 3/data/Random Tweets from Pakistan- Cleaned- Anonymous.csv"
if not path.is_file():
    raise FileNotFoundError("Run python download_data.py first.")
# The source contains malformed UTF-8 bytes. Preserve valid multilingual text
# and replace only invalid sequences, reporting their impact explicitly.
data = pd.read_csv(path, encoding="utf-8", encoding_errors="replace",
                   usecols=["full_text"])
print("Tweets with encoding replacement characters:",
      data["full_text"].fillna("").str.contains("\ufffd", regex=False).sum())
print("Rows:", len(data))
print("Missing tweets:", data["full_text"].isna().sum())
data[["full_text"]].head()

Tweets with encoding replacement characters: 6538
Rows: 202202
Missing tweets: 51


,full_text
0,تیرا لیڈر میرا لیڈر نواز شریف نواز شریف❤️ کیا...
1,"Happy birthday to my brother n boss , May you ..."
2,❤️❤️
3,`suspicious °jikook au jimin'in yaşadığı kasab...
4,Speaking of @SpiderMan... 😂 https://t.co/uGJ...


## 2. Preprocess the tweets

Remove entire hashtag tokens, standalone `RT` markers, URLs, mentions, and emojis as requested. Then lowercase and normalize whitespace. Tokenization keeps Unicode letter sequences; numbers and punctuation are omitted. Tweets are separate sentences, so bigrams never cross tweet boundaries.

In [3]:
import emoji

def clean_tweet(text):
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"(?<!\w)#\w+", " ", text)
    text = re.sub(r"(?<!\w)@\w+", " ", text)
    text = re.sub(r"\bRT\b", " ", text, flags=re.IGNORECASE)
    text = emoji.replace_emoji(text, replace=" ")
    return " ".join(text.lower().split())

def tokenize_tweet(text):
    return re.findall(r"[^\W\d_]+", clean_tweet(text), flags=re.UNICODE)

valid_text = data["full_text"].dropna().astype(str)
valid_text = valid_text[~valid_text.str.contains("\ufffd", regex=False)]
sentences = [tokenize_tweet(text) for text in valid_text]
sentences = [tokens for tokens in sentences if len(tokens) >= 2]
before = len(sentences)
# Identical cleaned tweets must not appear in both training and test sets.
sentences = list(dict.fromkeys(tuple(tokens) for tokens in sentences))
print("Tweets with at least two tokens:", before)
print("Repeated cleaned tweets removed:", before - len(sentences))
print("Unique usable tweets:", len(sentences))
print("Example tokens:", list(sentences[0])[:30])

Tweets with at least two tokens: 163971
Repeated cleaned tweets removed: 47272
Unique usable tweets: 116699
Example tokens: ['تیرا', 'لیڈر', 'میرا', 'لیڈر', 'نواز', 'شریف', 'نواز', 'شریف', 'کیا', 'آپکا', 'بھی', 'ھے']


## 3. Train a bigram MLE model

Use a reproducible 80/20 split before learning the vocabulary or counts. Tokens occurring fewer than twice in training are mapped to `<UNK>` so unseen test words can use an unknown-word category. This does not smooth unseen bigrams.

`padded_everygram_pipeline()` adds sentence boundary tokens and constructs the training n-grams. For bigrams, one `<s>` and one `</s>` mark each sentence. This is boundary padding, rather than padding every sequence to an equal length.

The bigram estimate is `P(next word | previous word) = count(previous, next) / count(previous)` using the training model's counts.

In [4]:
from sklearn.model_selection import train_test_split
from nltk.lm import MLE, Vocabulary
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.util import ngrams
from itertools import chain

train_sentences, test_sentences = train_test_split(sentences, test_size=0.2, random_state=42)
assert not set(train_sentences).intersection(test_sentences)
vocabulary = Vocabulary(chain.from_iterable(train_sentences), unk_cutoff=2)
vocabulary.update(["<s>", "</s>"] * 2)
train_ngrams, _ = padded_everygram_pipeline(2, train_sentences)
model = MLE(2, vocabulary=vocabulary)
model.fit(train_ngrams)
print("Training tweets:", len(train_sentences))
print("Test tweets:", len(test_sentences))
print("Vocabulary size including boundary and unknown tokens:", len(model.vocab))

Training tweets: 93359
Test tweets: 23340
Vocabulary size including boundary and unknown tokens: 49027


## 4. Evaluate the model

Perplexity is `2 ** mean(-log2 P(word | context))`; lower is better when comparing models on the same tokens. An unsmoothed MLE model assigns probability zero to unseen bigrams. A single zero-probability event makes test perplexity infinite. We report that honestly, together with the fraction of test bigrams that have nonzero probability. Test tokens are mapped through the training vocabulary.

In [5]:
import math

def evaluate_bigrams(sentences, model):
    total = unseen = 0
    negative_log_sum = 0.0
    for sentence in sentences:
        mapped = model.vocab.lookup(sentence)
        for previous, word in ngrams(pad_both_ends(mapped, n=2), 2):
            probability = model.score(word, [previous])
            total += 1
            if probability == 0:
                unseen += 1
            else:
                negative_log_sum -= math.log2(probability)
    perplexity = math.inf if unseen else 2 ** (negative_log_sum / total)
    return {"bigrams": total, "zero_probability_bigrams": unseen,
            "nonzero_coverage": (total - unseen) / total, "perplexity": perplexity}

results = pd.DataFrame({
    "train": evaluate_bigrams(train_sentences, model),
    "test": evaluate_bigrams(test_sentences, model)
}).T
results

,bigrams,zero_probability_bigrams,nonzero_coverage,perplexity
train,1808637.0,0.0,1.00000,79.757217
test,450783.0,119011.0,0.73599,inf


### Generate example tweets

Generate from a sentence-start token with fixed random seeds, stopping at `</s>` or 25 tokens. Generated text is a statistical demonstration, not a factual statement. A bigram model uses only one previous word and can produce incoherent sentences.

In [6]:
import random

def generate_tweet(seed, max_words=25):
    rng = random.Random(seed)
    context, words = "<s>", []
    for _ in range(max_words):
        word = model.generate(1, text_seed=[context], random_seed=rng)
        if word == "</s>":
            break
        if word != "<s>":
            words.append(word)
        context = word
    return " ".join(words)

for seed in (42, 43, 44):
    print(f"Seed {seed}: {generate_tweet(seed)}")

Seed 42: the <UNK> apnay apko name of them
Seed 43: <UNK> zinda baad hi nae parti hai exams
Seed 44: love pizza shop and does


## 5. Probability of the bigram `pakistan is`

The requested conditional probability is **P(is | pakistan)**. In NLTK, the predicted word is the first argument and the preceding context is the second.

In [7]:
context_count = model.counts[["pakistan"]].N()
pair_count = model.counts[["pakistan"]]["is"]
probability = model.score("is", ["pakistan"])
print("Count of context pakistan:", context_count)
print("Count of bigram pakistan is:", pair_count)
print("P(is | pakistan):", probability)
if context_count:
    assert math.isclose(probability, pair_count / context_count)

Count of context pakistan: 6746
Count of bigram pakistan is: 288
P(is | pakistan): 0.04269196560924993


## 6. Perplexity of `pakistan`

A single word does not specify its preceding bigram context. We show two explicit interpretations:

1. **Isolated-word (unigram) perplexity:** `1 / P(pakistan)`, using the unigram counts stored by the MLE model.
2. **One-word sentence perplexity:** evaluate the padded bigrams `(<s>, pakistan)` and `(pakistan, </s>)`.

These answer different questions and need not have the same value.

In [8]:
word_probability = model.score("pakistan")
word_perplexity = model.perplexity([("pakistan",)])
one_word_sentence = list(ngrams(pad_both_ends(["pakistan"], n=2), 2))
print("P(pakistan), without context:", word_probability)
print("Isolated-word perplexity:", word_perplexity)
print("One-word sentence bigrams:", one_word_sentence)
print("One-word sentence perplexity:", model.perplexity(one_word_sentence))
if word_probability:
    assert math.isclose(word_perplexity, 1 / word_probability)

P(pakistan), without context: 0.0035468003087283043
Isolated-word perplexity: 281.9442632671213
One-word sentence bigrams: [('<s>', 'pakistan'), ('pakistan', '</s>')]
One-word sentence perplexity: 34.17006405782672


## Conclusion

The notebook implements the five requested cleaning operations, trains a bigram MLE model, evaluates held-out tweets, generates text, and calculates the requested probability and perplexity. Infinite held-out perplexity is an expected limitation of unsmoothed MLE when test bigrams are unseen; it is not replaced with a fabricated finite score. Smoothing would be a possible later extension.

### Observed results

After removing missing or damaged text, short tweets, and duplicate token sequences, **116,699 tweets** remain: **93,359 training** and **23,340 test** tweets. The source had **6,538 tweets containing encoding replacement characters**, which were excluded.

- Training perplexity: **79.76**.
- Held-out perplexity: **infinite**; **119,011 of 450,783** test bigrams have zero probability (nonzero coverage **73.60%**).
- `P(is | pakistan) = 288 / 6746`, approximately **0.04269**.
- Isolated-word perplexity of `pakistan`: **281.94**.
- Padded one-word sentence perplexity: **34.17**.
